# Model and Evaluation for Approval Predict 

## Objective:

* Answer business requirement 2:
The client aims to offer a guide for potential applicants by identifying the influential factors that contribute to loan approval. These insights will be used to recommend specific improvements for applicants and guide them to increase their chances of having a loan approved.

* Fit and evaluate classification models to predict loan approval outcomes. Classification will directly predict whether a loan is approved.

## Inputs
* outputs/datasets/collection/loan_approved.csv
* Instructions on which variables to use for data cleaning and feature engineering. They are found in each respective notebook.

## Outputs
* Modeling pipeline
* Feature importance plot

## Add imports

In [ ]:
import os
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from feature_engine.transformation import (
    BoxCoxTransformer,
    YeoJohnsonTransformer,
)
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    make_scorer,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
sns.set(style="whitegrid")

## Change Working Directory

In [ ]:
current_dir = os.getcwd()
current_dir

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

In [ ]:
current_dir = os.getcwd()
current_dir

## Load data

we load the dataset with name and city removed. The dataframe for the target loan_approved is converted to an iteger. The loan_to_income feature is added.

In [ ]:
root = current_dir
file_path = (
    Path(root)
    / "outputs"
    / "datasets"
    / "collection"
    / "loan_approval.csv"
)

if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {file_path}"
    )

df = pd.read_csv(file_path).drop(['name', 'city'], axis=1)
df['loan_approved'] = df['loan_approved'].astype(int)
df['loan_to_income'] = df['loan_amount'] / df['income']
df.head(3)
print(df.shape)

## ML Pipeline with all data
ML pipeline for Data Cleaning and Feature Engineering.

In [ ]:
final_features = [
    'points',
    'loan_to_income',
    'credit_score',
    'years_employed',
    'loan_amount'
]

X = df[final_features].copy()
y = df['loan_approved'].copy()


def approval_pipeline():
    """
    Create a machine learning pipeline that applies Yeo-Johnson and Box-Cox
    transformations, robust scaling, and a classification model for predicting
    loan approval.
    """
    pipeline = Pipeline([
        ('yeojohnson', YeoJohnsonTransformer(variables=['points'])),
        ('boxcox', BoxCoxTransformer(variables=['loan_to_income'])),
        ('scaler', RobustScaler()),
        ('model', model)
    ])
    return pipeline

## ML Pipeline for Modelling and Hyperparameter Optimisation

In [ ]:
def PipelineClf(model):
    """
    Builds a pipeline for hyperparameter optimisation that includes
    feature transformations, robust scaling and a classifier model.
    """
    return Pipeline([
        ("yeojohnson", YeoJohnsonTransformer(variables=['points'])),
        ("boxcox", BoxCoxTransformer(variables=['loan_to_income'])),
        ("scaler", RobustScaler()),
        ("model", model),
    ])

A custom class, HyperparameterOptimizationSearch, was created by the CodeInstitute to perform grid searches across selected parameter spaces for each model.

In [ ]:
class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = PipelineClf(self.models[key])
            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring, )
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]
        return df[columns], self.grid_searches

## Split Train and Test Set

We split the dataset into a train (80%) and test set (20%). The dataset was trialed with 70/30 but did not add any added value to the findings.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Train/Test Split Shapes:")
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

## Handle Target Imbalance

We check train set target distribution using Synthetic Minority Oversampling Technique (SMOTE). This creates synthetic samples of the minority class.

In [ ]:
y_train.value_counts().plot(kind='bar', title='Loan Approved Distribution')
plt.show()

Loan approvals contain fewer positive cases. To allow the model to learn approval patterns better, we apply SMOTE oversampling on the training set.

Use SMOTE (Synthetic Minority Oversampling Technique) to balance Train Set target

In [ ]:
smote = SMOTE(sampling_strategy='minority', random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("After SMOTE:", X_train_bal.shape, y_train_bal.shape)

y_train_bal.value_counts().plot(
    kind="bar",
    title="Balanced Loan Approved Distribution",
)
plt.show()

## Grid Search CV - Sklearn

This step identifies the best-performing model before detailed hyperparameter tuning.

In [ ]:
models_quick_search = {
    "LogisticRegression": LogisticRegression(random_state=42),
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=42),
    "RandomForestClassifier": RandomForestClassifier(random_state=42),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=42),
    "ExtraTreesClassifier": ExtraTreesClassifier(random_state=42),
    "AdaBoostClassifier": AdaBoostClassifier(random_state=42),
    "XGBClassifier": XGBClassifier(random_state=42)
}

params_quick_search = {
    "LogisticRegression": {},
    "DecisionTreeClassifier": {},
    "RandomForestClassifier": {},
    "GradientBoostingClassifier": {},
    "ExtraTreesClassifier": {},
    "AdaBoostClassifier": {},
    "XGBClassifier": {},
}

## Quick GridSearch CV - Binary Classifier
This selects the best algorithm using results from the quick search. We review the best performing model based on our scoring metric (Recall on the approved class).

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_quick_search,
    params=params_quick_search
)

search.fit(
    X_train_bal,
    y_train_bal,
    scoring=make_scorer(recall_score, pos_label=1),
    cv=5,
    n_jobs=-1,
    verbose=1
)

## Hyperparameter search

In [ ]:
search = HyperparameterOptimizationSearch(
    models_quick_search,
    params_quick_search,
)

search.fit(
    X_train_bal, y_train_bal,
    scoring=make_scorer(recall_score, pos_label=1),
    cv=5, n_jobs=-1, verbose=1
)

The results of the grid search sorts the best estimator by its mean score.
The top 3 estimators was DecisionTreeClassifier, RandomForestClassifier and GradientBoostingClassifier.

In [ ]:
results_summary, search_pipelines = search.score_summary(sort_by='mean_score')
results_summary

Define the best clf pipeline

In [ ]:
best_model = results_summary.iloc[0, 0]
pipeline_clf = search.grid_searches[best_model].best_estimator_
pipeline_clf

The following functions are from Code Institute and used to evaluate the machine learning classifier by producing a confusion matrix and a classification report for both the training and test datasets.

In [ ]:
def confusion_matrix_and_report(X, y, pipeline, label_map):
    prediction = pipeline.predict(X)

    print('---  Confusion Matrix  ---')
    print(pd.DataFrame(
        confusion_matrix(y_true=prediction, y_pred=y),
        columns=[["Actual " + sub for sub in label_map]],
        index=[["Prediction " + sub for sub in label_map]]
    ))
    print("\n")

    print('---  Classification Report  ---')
    print(classification_report(y, prediction, target_names=label_map), "\n")


def clf_performance(X_train, y_train, X_test, y_test, pipeline, label_map):
    print("#### Train Set #### \n")
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

    print("#### Test Set ####\n")
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map)

In [ ]:
print(f"Quick-search best model: {best_model}")

clf_performance(
    X_train_bal, y_train_bal,
    X_test, y_test,
    pipeline_clf,
    label_map=["Rejected", "Approved"]
)

Both the train and test set achieved a performace of 100% scores in Prescision, Recall and F1-Score. While these results meet the appropriate model success metrics, it may be overreliant on the `points` feature. Therefore, this stops the model from learning from the other features, which is required for business requirement 2.

Due to these results no hyperparameter optimisation is required. However, we will create a more robust model with `points` feature being removed.

We use a bar plot to show which features influence the model's predictions the most.

In [ ]:
top_features = X_train.columns

df_feature_importance = (pd.DataFrame(data={
    'Feature': top_features,
    'Importance': pipeline_clf['model'].feature_importances_})
    .sort_values(by='Importance', ascending=False)
)

df_feature_importance.plot(kind="bar", x="Feature", y="Importance")
plt.show()

The Decision Tree Classifier feature importance bar plot shows points is the key approval driver.

## Save Feature Importance Plot

The following files will be saved:
* Feature plot

In [ ]:
version = "v1"
file_path = f"outputs/ml_pipeline/approval_prediction/{version}"

os.makedirs(file_path, exist_ok=True)
print("Saving to:", file_path)

### Feature Importance Plot

Save visualisation showing points feature dominance.

In [ ]:
df_feature_importance.plot(kind="bar", x="Feature", y="Importance")
plt.savefig(f"{file_path}/features_importance.png", bbox_inches="tight")
plt.show()

print("Feature importance plot saved")

## Conclusions

The DecisionTreeClassifier achieved **perfect 100% scores** (precision, recall and F1-score) on both training and test sets without any hyperparameter optimisation.

However, this model is not suitable due to complete reliance on the `points` feature and only can provide 1 feature that is contributing the overall approval outcome. Thus not sufficiently answering the second business requirement.


## Next Steps

To build a model that meets both business requirements by removing the `points` feature and train models using the features of `credit_score`, `income`, `years_employed`, `loan_amount` and `loan_to_income`.
